# Протей на TinyStories — Kaggle**Перед запуском:** справа Settings → Accelerator → **GPU T4 x2**, Internet → **On**.Без интернета не скачается датасет, без GPU прогон займёт недели.Отличие от Colab: Kaggle рвёт сессию через ~12 ч, поэтому у обучения есть`--max-hours` — оно выйдет само, сохранив состояние. Результаты пишем в`/kaggle/working` (это сохраняется между сессиями в рамках версии ноутбука).

## 1. GPU и код

In [ ]:
import torchassert torch.cuda.is_available(), "Settings -> Accelerator -> GPU T4"print(torch.cuda.get_device_name(0))!pip install -q datasets!git clone -q https://github.com/maleshovivan23-creator/-.git /kaggle/working/ultranet 2>/dev/null || true%cd /kaggle/working/ultranet!git checkout -q arena/01a0b93c-repo && git pull -qimport sys; sys.path.insert(0, '/kaggle/working/ultranet')

## 2. Данные~10 минут. Токенизация быстрая (15M символов/с), основное время — скачивание.

In [ ]:
!python colab/prepare_data.py --vocab 4096 --out /kaggle/working/data

## 3. Проба: 200 шагов**Не пропускайте.** Смотрим одно: стартовый loss должен быть ≈ ln(4096) = **8.3**.* 8.3 и падает к ~5 — всё исправно, идём дальше* 200+ — сломана инициализация* 3–4 — словарь меньше заявленного, проверьте `prepare_data`

In [ ]:
!python colab/train.py --steps 200 \    --train-bin /kaggle/working/data/train.bin \    --val-bin /kaggle/working/data/val.bin \    --tokenizer /kaggle/working/data/tokenizer.json \    --out /kaggle/working/probe

## 4. Полный прогон`--max-hours 11` — выйдет сам до того, как Kaggle убьёт сессию.Если прогон не закончился, **просто запустите эту ячейку снова**: продолжитс последнего чекпоинта.

In [ ]:
!python colab/train.py \    --steps 60000 --batch 32 --seq 256 \    --dim 256 --layers 6 --heads 8 --lr 6e-4 \    --max-hours 11 \    --train-bin /kaggle/working/data/train.bin \    --val-bin /kaggle/working/data/val.bin \    --tokenizer /kaggle/working/data/tokenizer.json \    --out /kaggle/working/tinystories-16m

## 5. Сэмплы

In [ ]:
import torch, syssys.path.insert(0, '/kaggle/working/ultranet')sys.path.insert(0, '/kaggle/working/ultranet/colab')from ultranet.models import GPTConfigfrom ultranet.torch_port import TorchGPTfrom ultranet.tokenizer import BPETokenizerfrom train import sampleOUT = '/kaggle/working/tinystories-16m'tok = BPETokenizer.load('/kaggle/working/data/tokenizer.json')st = torch.load(f'{OUT}/last.pt', map_location='cuda', weights_only=False)model = TorchGPT(GPTConfig(**st['gcfg'])).cuda(); model.load_state_dict(st['model'])print(f"шаг {st['step']}\n")for p in ["Once upon a time", "Lily found a", "The little boy was very"]:    print('>', sample(model, tok, p, 120, 'cuda', st['gcfg']['block_size'], temperature=0.8))    print()

## 6. Кривая loss

In [ ]:
import json, matplotlib.pyplot as pltrows = [json.loads(l) for l in open(f'{OUT}/log.jsonl')]plt.figure(figsize=(9,4))plt.plot([r['step'] for r in rows], [r['loss'] for r in rows])plt.xlabel('шаг'); plt.ylabel('loss'); plt.grid(alpha=.3); plt.show()print(f"{rows[-1]['tok_s']:,} токенов/с | последний loss {rows[-1]['loss']}")